# Bregman Proximal Gradient Method

This code outlines the procedure for Lyapunov function discovery for the Bregman Proximal Gradient Method (BPGM). We study composite convex minimization $\min_x F(x):=f(x)+g(x)$, where $f$ is $1/\alpha$-relatively smooth with respect to a convex differentiable Bregman kernel $h$, equivalently $h-\alpha f$ is convex. The Bregman divergence is $D_h(x,y)=h(x)-h(y)-\langle \nabla h(y),x-y\rangle$, and the performance metric is $F(x_N)-F(x_\diamond)$ under $D_h(x_\diamond,x_0)\le R$.


## Import the required libraries

In [1]:
import pepflow as pf
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from itertools import product
from IPython.display import display, Math

## Define the functions and parameters

In [2]:
stepsize = pf.Parameter("stepsize")
R = pf.Parameter("R")

f = pf.ConvexFunction(is_basis=True, tags=["f"])
g = pf.ConvexFunction(is_basis=True, tags=["g"])
h = pf.ConvexFunction(is_basis=True, tags=["h"])
F = (f + g).add_tag("F")

## Write a function to return the PEPContext for BPGM

In [ ]:
def relative_smoothness_ineq(h_func, f_func, x_i, x_j, alpha):
    return (
        h_func(x_j)
        - h_func(x_i)
        - alpha * (f_func(x_j) - f_func(x_i))
        + (h_func.grad(x_j) - alpha * f_func.grad(x_j)) * (x_i - x_j)
    )


def relative_smoothness_name(row_tag, col_tag):
    return f"relative_smoothness:{row_tag},{col_tag}"


def make_ctx_bpgm(ctx_name: str, N: int | sp.Integer) -> pf.PEPContext:
    ctx = pf.PEPContext(ctx_name).set_as_current()
    x = pf.Vector(is_basis=True, tags=["x_0"])
    pf.Vector(is_basis=True, tags=["x_diamond"])

    for k in range(int(N)):
        grad_f_x = f.grad(x)
        grad_h_x = h.grad(x)

        x_next = pf.Vector(is_basis=True, tags=[f"x_{k + 1}"])
        grad_h_next = h.grad(x_next)
        grad_g_next = (grad_h_x - grad_h_next) / stepsize - grad_f_x
        grad_g_next.math_expr.expr_str = f"grad_g({x_next})"
        g.add_point_with_grad_restriction(x_next, grad_g_next)
        x = x_next

    return ctx


def make_pep_bpgm(ctx_name: str, N: int | sp.Integer):
    ctx = make_ctx_bpgm(ctx_name, N)
    pb = pf.PEPBuilder(ctx)
    point_tags = [f"x_{i}" for i in range(int(N) + 1)] + ["x_diamond"]
    points = [ctx[tag] for tag in point_tags]
    for x_i, x_j in product(points, points):
        if x_i == x_j:
            continue
        pb.add_initial_constraint(
            relative_smoothness_ineq(h, f, x_i, x_j, stepsize).le(
                0,
                name=relative_smoothness_name(str(x_i), str(x_j)),
            )
        )
    pb.add_initial_constraint(
        (-h.interp_ineq("x_diamond", "x_0", pep_context=ctx)).le(
            R,
            name="initial_condition",
        )
    )
    pb.set_relaxed_constraints(
        [
            f"g:{point_tag},x_diamond"
            for point_tag in point_tags
            if point_tag != "x_diamond"
        ]
    )
    pb.set_performance_metric(F(ctx[f"x_{N}"]) - F(ctx["x_diamond"]))
    return ctx, pb

## Numerical evidence showing that BPGM converges at the rate $F(x_N)-F(x_\diamond)\le \frac{R}{\alpha N}$

In [ ]:
N_max = 5
stepsize_value = 1
R_value = 1

opt_values = []
for N_i in range(1, N_max + 1):
    ctx_i, pb_i = make_pep_bpgm(ctx_name=f"ctx_plt_{N_i}", N=N_i)
    result_i = pb_i.solve(resolve_parameters={"stepsize": stepsize_value, "R": R_value})
    opt_values.append(result_i.opt_value)

iters = np.arange(1, N_max + 1)
cont_iters = np.arange(1, N_max + 0.01, 0.01)
plt.plot(
    cont_iters,
    R_value / (stepsize_value * cont_iters),
    "r-",
    label="Analytical bound $R/(\\alpha N)$",
)
plt.scatter(iters, opt_values, color="blue", marker="o", label="Numerical values")
plt.xlabel("N")
plt.ylabel(r"Worst-case $F(x_N)-F(x_\diamond)$")
plt.legend()

## Verification of convergence of BPGM

In [ ]:
N = sp.S(4)
stepsize_value = sp.S(1)
R_value = sp.S(1)

ctx_prf, pb_prf = make_pep_bpgm("ctx_prf", N)
result = pb_prf.solve(resolve_parameters={"stepsize": stepsize_value, "R": R_value})
print(result.opt_value)

### Store the solver output and matrix labels used below

In [ ]:
def display_label(label: str) -> str:
    return str(label).replace("x_diamond", r"x_\diamond")


def display_labels(labels) -> list[str]:
    return [display_label(label) for label in labels]


def relative_dual_matrix(result, N):
    names = [f"x_{i}" for i in range(int(N) + 1)] + ["x_diamond"]
    matrix = np.zeros((len(names), len(names)))
    for i, row in enumerate(names):
        for j, col in enumerate(names):
            if row != col:
                matrix[i, j] = (
                    result.get_dual_value(relative_smoothness_name(row, col)) or 0.0
                )
    return pf.MatrixWithNames(matrix, row_names=names, col_names=names)


tau_sol = result.dual_var_manager.dual_value("initial_condition")

f_lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(f)
g_lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(g)
h_lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(h)
rel_lamb_sol = relative_dual_matrix(result, N)

S_sol = result.get_gram_dual_matrix()

In [ ]:
print("f rows:", f_lamb_sol.row_names)
print("g rows:", g_lamb_sol.row_names)
print("rel rows:", rel_lamb_sol.row_names)
print("h rows:", h_lamb_sol.row_names)

---

## Step 1. Propose a candidate Lyapunov function

### Let
$$
\begin{aligned}
\mathcal I^f_0&=\emptyset,
&\mathcal I^f_k&=\{(\diamond,j):j=0,\ldots,k-1\}\cup\{(j-1,j):j=1,\ldots,k-1\},\\
\mathcal I^g_0&=\emptyset,
&\mathcal I^g_k&=\{(\diamond,j):j=1,\ldots,k\}\cup\{(j-1,j):j=1,\ldots,k-1\},\\
\mathcal I^{h-\alpha f}_0&=\emptyset,
&\mathcal I^{h-\alpha f}_k&=\{(j,j-1):j=1,\ldots,k\}\cup\{(j-1,j):j=1,\ldots,k-1\}
\end{aligned}
$$
### for $k=1,\ldots,N$, and let $\mathcal J_k=\emptyset$ for $k=0,\ldots,N$.


In [ ]:
V_raw = [pf.Scalar.zero()]
partial_sum = pf.Scalar.zero()

for j in range(1, int(N) + 1):
    x_prev_tag = f"x_{j - 1}"
    x_curr_tag = f"x_{j}"

    if j >= 2:
        partial_sum += (
            stepsize_value
            * sp.S(j - 1)
            * f.interp_ineq(
                x_prev_tag,
                x_curr_tag,
                pep_context=ctx_prf,
            )
        )
    partial_sum += stepsize_value * f.interp_ineq(
        "x_diamond",
        x_prev_tag,
        pep_context=ctx_prf,
    )

    if j >= 2:
        partial_sum += (
            stepsize_value
            * sp.S(j - 1)
            * g.interp_ineq(
                x_prev_tag,
                x_curr_tag,
                pep_context=ctx_prf,
            )
        )
    partial_sum += stepsize_value * g.interp_ineq(
        "x_diamond",
        x_curr_tag,
        pep_context=ctx_prf,
    )

    if j >= 2:
        partial_sum += sp.S(j - 1) * relative_smoothness_ineq(
            h,
            f,
            ctx_prf[x_prev_tag],
            ctx_prf[x_curr_tag],
            stepsize,
        )
    partial_sum += sp.S(j) * relative_smoothness_ineq(
        h,
        f,
        ctx_prf[x_curr_tag],
        ctx_prf[x_prev_tag],
        stepsize,
    )

    V_raw.append(partial_sum)

print("V_raw contains:", [f"V_raw_{k}" for k in range(len(V_raw))])

## Step 2. Check for admissibility

### Sufficiency is immediate because the sets at $k=N$ contain the adjacent interpolation constraints and all diamond interpolation constraints kept by the sparse certificate. We first inspect the raw partial sums, for which $V^{raw}_0=0$.
### Then we translate them by the initial-condition term $D_h(x_\diamond,x_0)=-I_h(x_\diamond,x_0)$:
$$
V_k = V^{raw}_k + D_h(x_\diamond,x_0).
$$
### This is the BPGM analogue of adding the initial-distance term in OGM: the added quantity is independent of $k$ and is controlled by the initial condition $D_h(x_\diamond,x_0)\le R$.

In [ ]:
pm = pf.ExpressionManager(ctx_prf, resolve_parameters={"stepsize": stepsize_value})
scalar_labels = display_labels(ctx_prf.basis_scalars())
rank_tolerance = 1e-7

for k, V in enumerate(V_raw):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(
        V_eval.inner_prod_coords.astype(float), tol=rank_tolerance
    )
    func_coords = V_eval.func_coords.astype(float)
    support = [scalar_labels[i] for i, val in enumerate(func_coords) if abs(val) > 1e-7]
    print(f"V_raw_{k}: rank={rank}, nonzero function coordinates={support}")

initial_bregman = -h.interp_ineq("x_diamond", "x_0", pep_context=ctx_prf)
lyap = [V + initial_bregman for V in V_raw]

print("\nAfter adding the initial Bregman term:")
for k, V in enumerate(lyap):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(
        V_eval.inner_prod_coords.astype(float), tol=rank_tolerance
    )
    func_coords = V_eval.func_coords.astype(float)
    support = [scalar_labels[i] for i, val in enumerate(func_coords) if abs(val) > 1e-7]
    print(f"V_{k}: rank={rank}, nonzero function coordinates={support}")

## Step 3. Build a set of special candidate vectors

In [ ]:
special_vectors = []

for k in range(len(lyap)):
    h_grad_k = ctx_prf.get_triplet_by_point_tag(f"x_{k}", func=h).expand()[2]
    special_vectors.append([h_grad_k, ctx_prf[f"x_{k}"] - ctx_prf["x_diamond"]])
    print(f"V_{k}:", [f"grad_h(x_{k})", rf"x_{k}-x_\diamond"])

## Step 4. Find a basis of $\mathbf{V}_k$ within the special candidate vectors

### For each $V_k$, display the function-value coefficients and the quadratic coefficient matrix on $[\nabla h(x_k), x_k-x_\diamond]$.

In [ ]:
from pepflow.lyapunov_utils import find_symmetric_coefficient_matrix

for k, V in enumerate(lyap):
    C_k = find_symmetric_coefficient_matrix(
        V,
        special_vectors[k],
        pep_context=ctx_prf,
        resolve_parameters=pm.resolve_parameters,
    )
    print(f"V_{k}: function-value coefficients")
    pf.pprint_labeled_vector(
        pm.eval_scalar(V).func_coords.astype(float),
        scalar_labels,
    )
    print(f"V_{k}: quadratic coefficients")
    labels_k = [f"grad_h(x_{k})", rf"x_{k}-x_\diamond"]
    pf.pprint_labeled_matrix(C_k, labels_k, labels_k)

### We can hypothesize the form
$$
V_k = a_k\bigl(F(x_k)-F(x_\diamond)\bigr)+b_kD_h(x_\diamond,x_k),
$$
### equivalently
$$
V_k = a_k\bigl(F(x_k)-F(x_\diamond)\bigr)
+b_k\Bigl(h(x_\diamond)-h(x_k)-\langle \nabla h(x_k),x_\diamond-x_k\rangle\Bigr).
$$
### The function-value coefficient vector identifies the first term, and the rank-2 decomposition on $[\nabla h(x_k),x_k-x_\diamond]$ identifies the Bregman term.

---

## Step 5. Analytic proof

### Write the closed-form interpolation weights suggested by the certificate.

In [ ]:
def internal_label(tag: str) -> str:
    return str(tag).replace(r"x_\diamond", "x_diamond")


def idx(tag, N=N):
    tag = internal_label(tag)
    if tag == "x_diamond":
        return int(N) + 1
    return int(tag.split("_")[1])


def f_lamb(ri, ci, N=N):
    ri = internal_label(ri)
    ci = internal_label(ci)
    if ri == "x_diamond" and ci.startswith("x_"):
        j = idx(ci, N)
        if 0 <= j <= int(N) - 1:
            return sp.S(1) / N
    if ri.startswith("x_") and ci.startswith("x_"):
        i, j = idx(ri, N), idx(ci, N)
        if 0 <= i < int(N) and j == i + 1:
            return sp.S(i) / N
    return sp.S(0)


def g_lamb(ri, ci, N=N):
    ri = internal_label(ri)
    ci = internal_label(ci)
    if ri == "x_diamond" and ci.startswith("x_"):
        j = idx(ci, N)
        if 1 <= j <= int(N):
            return sp.S(1) / N
    if ri.startswith("x_") and ci.startswith("x_"):
        i, j = idx(ri, N), idx(ci, N)
        if 1 <= i < int(N) and j == i + 1:
            return sp.S(i) / N
    return sp.S(0)


def rel_lamb(ri, ci, N=N):
    ri = internal_label(ri)
    ci = internal_label(ci)
    if ri.startswith("x_") and ci.startswith("x_"):
        i, j = idx(ri, N), idx(ci, N)
        if 0 <= i < int(N) and j == i + 1:
            return sp.S(i) / (stepsize_value * N)
        if 1 <= i <= int(N) and j == i - 1:
            return sp.S(i) / (stepsize_value * N)
    return sp.S(0)


def h_lamb(ri, ci, N=N):
    ri = internal_label(ri)
    ci = internal_label(ci)
    if ri == "x_diamond" and ci == f"x_{int(N)}":
        return sp.S(1) / (stepsize_value * N)
    return sp.S(0)


for label, lamb, sol in [
    ("f", f_lamb, f_lamb_sol),
    ("g", g_lamb, g_lamb_sol),
    ("h-alpha*f", rel_lamb, rel_lamb_sol),
    ("h", h_lamb, h_lamb_sol),
]:
    print(label)
    pf.pprint_labeled_matrix(
        lamb,
        display_labels(sol.row_names),
        display_labels(sol.col_names),
    )

### Determine the Lyapunov coefficients by solving the one-step identity
$$
V_k-V_{k-1}=\alpha(k-1)I_f(x_{k-1},x_k)+\alpha I_f(x_\diamond,x_{k-1})
+\alpha(k-1)I_g(x_{k-1},x_k)+\alpha I_g(x_\diamond,x_k)
+(k-1)I_{h-\alpha f}(x_{k-1},x_k)+kI_{h-\alpha f}(x_k,x_{k-1}).
$$


In [ ]:
alpha_param = pf.Parameter("alpha")
alpha_sym = sp.Symbol("alpha", positive=True)
k_sym = sp.Symbol("k", integer=True, positive=True)

ctx_step = pf.PEPContext("step").set_as_current()
f_step = pf.ConvexFunction(is_basis=True, tags=["f_{step}"])
g_step = pf.ConvexFunction(is_basis=True, tags=["g_{step}"])
h_step = pf.ConvexFunction(is_basis=True, tags=["h_{step}"])
F_step = (f_step + g_step).add_tag("F_{step}")

x_prev = pf.Vector(is_basis=True, tags=["x_{k-1}"])
pf.Vector(is_basis=True, tags=["x_diamond"])
h_step(ctx_step["x_diamond"])
f_step(x_prev)
g_step(x_prev)
h_step(x_prev)

x_curr = pf.Vector(is_basis=True, tags=["x_k"])
f_step(x_curr)
h_step(x_curr)
grad_g_curr = (h_step.grad(x_prev) - h_step.grad(x_curr)) / alpha_param - f_step.grad(
    x_prev
)
grad_g_curr.math_expr.expr_str = f"grad_g({x_curr})"
g_step.add_point_with_grad_restriction(x_curr, grad_g_curr)

a_prev = pf.Parameter("a_{k-1}")
a_curr = pf.Parameter("a_k")
b_prev = pf.Parameter("b_{k-1}")
b_curr = pf.Parameter("b_k")

V_prev = a_prev * (F_step(x_prev) - F_step(ctx_step["x_diamond"]))
V_prev -= b_prev * h_step.interp_ineq("x_diamond", "x_{k-1}", pep_context=ctx_step)

V_curr = a_curr * (F_step(x_curr) - F_step(ctx_step["x_diamond"]))
V_curr -= b_curr * h_step.interp_ineq("x_diamond", "x_k", pep_context=ctx_step)

rhs_step = (
    alpha_param
    * (k_sym - 1)
    * f_step.interp_ineq("x_{k-1}", "x_k", pep_context=ctx_step)
)
rhs_step += alpha_param * f_step.interp_ineq(
    "x_diamond", "x_{k-1}", pep_context=ctx_step
)
rhs_step += (
    alpha_param
    * (k_sym - 1)
    * g_step.interp_ineq("x_{k-1}", "x_k", pep_context=ctx_step)
)
rhs_step += alpha_param * g_step.interp_ineq("x_diamond", "x_k", pep_context=ctx_step)
rhs_step += (k_sym - 1) * relative_smoothness_ineq(
    h_step, f_step, x_prev, x_curr, alpha_param
)
rhs_step += k_sym * relative_smoothness_ineq(
    h_step, f_step, x_curr, x_prev, alpha_param
)

ak_prev_sym, ak_sym, bk_prev_sym, bk_sym = sp.symbols("a_{k-1} a_k b_{k-1} b_k")
unknowns = (ak_prev_sym, ak_sym, bk_prev_sym, bk_sym)
resolve_coeffs = {
    "alpha": alpha_sym,
    "a_{k-1}": ak_prev_sym,
    "a_k": ak_sym,
    "b_{k-1}": bk_prev_sym,
    "b_k": bk_sym,
}

pm_coeffs = pf.ExpressionManager(ctx_step, resolve_parameters=resolve_coeffs)
coeff_residual = pm_coeffs.eval_scalar(V_curr - V_prev - rhs_step, sympy_mode=True)
eqs = list(np.array(coeff_residual.inner_prod_coords, dtype=object).reshape(-1))
eqs += list(np.array(coeff_residual.func_coords, dtype=object).reshape(-1))
eqs = [sp.factor(sp.together(eq)) for eq in eqs if sp.simplify(eq) != 0]

sol_tuple = next(iter(sp.linsolve(eqs, unknowns)))
sol_dict = {
    unknown: sp.factor(sp.together(value))
    for unknown, value in zip(unknowns, sol_tuple)
}
display(Math(sp.latex(sol_dict)))

residual_after_sol = [sp.simplify(eq.subs(sol_dict)) for eq in eqs]
print("coefficient system solved:", all(eq == 0 for eq in residual_after_sol))
print("step identity zero:", all(eq == 0 for eq in residual_after_sol))

### Since the one-step identity also covers $k=1$, summing it gives that $V_k=\alpha k(F(x_k)-F(x_\diamond))+D_h(x_\diamond,x_k)$ is nonincreasing. With $V_0=D_h(x_\diamond,x_0)\le R$ and convexity of $h$, this yields $F(x_N)-F(x_\diamond)\le R/(\alpha N)$.